# 🎵 AI Song Generator - Colab 一键部署

This notebook will clone the AI song-generation repository, install dependencies, set up the ACE-Step 1.5 model, and launch an interactive Gradio app to generate songs from lyrics and style descriptions.

**Note:** GPU required. On first run, please monitor VRAM usage. If you encounter out-of-memory errors, use the "短版 Demo" (short version) option or reduce `infer_step` in the code.

## Step 1: Clone Repository and Enter Directory

In [ ]:
import os
import subprocess

# Clone the repository
# TODO: 替换为你的真实仓库地址（若仓库私有需先配置访问权限）
subprocess.run("git clone https://github.com/wisers/music.git", shell=True, check=True)

# Change to repo directory
os.chdir("/content/music")
print(f"Current directory: {os.getcwd()}")
print("Repository cloned and ready.")

## Step 2: Install Dependencies from requirements.txt

In [ ]:
# Install dependencies
subprocess.run("pip install -r requirements.txt", shell=True, check=True)
print("\n✓ Dependencies installed successfully.")

## Step 3: Install ACE-Step 1.5 Model and Weights

⚠️ **IMPORTANT:** Follow ACE-Step 1.5's official installation from their README:
- GitHub: [ACE-Step](https://github.com/bluesunrise/ACEStep) (adjust URL if needed)
- Install via: `pip install acestep` or `pip install git+https://github.com/bluesunrise/ACEStep.git`
- Weights (~2-3GB) will auto-download on first run

**⚠️ VRAM Alert:** ACE-Step 1.5 typically requires 8-24GB VRAM depending on inference steps.
- If you see `RuntimeError: CUDA out of memory`, try:
  - Restart the runtime and use "短版 Demo" (45 sec) instead of "完整歌曲" (210 sec)
  - Reduce `infer_step` from 27 to 15-20 in the smoke test cell below
  - Or use a GPU with more VRAM (e.g., A100 if available)

In [ ]:
# Install ACE-Step 1.5 - adjust URL per official README if needed
# Official: https://github.com/bluesunrise/ACEStep
subprocess.run("pip install acestep", shell=True, check=True)

print("\n✓ ACE-Step 1.5 installed.")
print("Note: Model weights will be downloaded on first use (~2-3GB).")

## Step 4: Set Anthropic API Key

Store your Anthropic API key in Colab Secrets:
1. Open the 🔑 **Secrets** panel on the left sidebar
2. Create a new secret named `ANTHROPIC_API_KEY`
3. Paste your API key from [console.anthropic.com](https://console.anthropic.com)
4. Run the cell below to load it

In [ ]:
from google.colab import userdata

# Retrieve API key from Colab Secrets
api_key = userdata.get('ANTHROPIC_API_KEY')
os.environ['ANTHROPIC_API_KEY'] = api_key

print("✓ ANTHROPIC_API_KEY set from Colab Secrets")
print(f"  (Key length: {len(api_key)} chars, starts with 'sk-ant-...'? {api_key.startswith('sk-ant-')})") if api_key else print("  (Warning: API key not found in secrets)")

## Step 5: Smoke Test - Generate a Short Song

This test ensures the entire pipeline (LLM planning, lyric structure, song generation) works end-to-end.
Expected output: a WAV file saved to `outputs/song.wav`

In [ ]:
# Change back to repo root for imports
os.chdir("/content/music")

from src.pipeline import make_song

print("Running smoke test with short lyrics and style...")
print("-" * 60)

test_lyrics = """早上起来看到你的笑脸，
阳光洒在窗口，
我们一起唱歌，
永远不分开。"""

test_style = "female vocal, soft pop, morning, gentle"

try:
    result = make_song(test_lyrics, test_style, length="short", seed=42)
    
    song_path = result["song"]
    structured_lyrics = result["structured_lyrics"]
    spec = result["spec"]
    
    print(f"✓ Smoke test passed!")
    print(f"  Song saved to: {song_path}")
    print(f"  Song size: {os.path.getsize(song_path) / 1024 / 1024:.1f} MB")
    print(f"\nStructured Lyrics (first 200 chars):")
    print(f"  {structured_lyrics[:200]}...")
    print(f"\nSong Spec:")
    print(f"  Genre: {spec.genre}")
    print(f"  Mood: {spec.mood}")
    print(f"  BPM: {spec.bpm}")
    print("-" * 60)
    print("✓ All systems ready for app launch!")
    
except Exception as e:
    print(f"✗ Smoke test failed: {e}")
    print(f"\nTroubleshooting tips:")
    print(f"  1. Ensure ANTHROPIC_API_KEY is set correctly (step 4)")
    print(f"  2. Check VRAM usage - if OOM, restart and use 'short' length only")
    print(f"  3. Verify ACE-Step 1.5 installation: 'pip show acestep'")
    raise

## Step 6: Launch Gradio App with Public Share Link

This cell starts the interactive web interface. You'll receive a public share link (valid for 72 hours) that you can use to generate songs.

**Usage:**
1. Enter your lyrics in the text area
2. Describe the desired feeling/style (e.g., "happy, upbeat, summer")
3. Choose "短版 Demo" (45 sec) or "完整歌曲" (210 sec) from advanced settings
4. Optional: enter a seed for reproducibility
5. Click "帮我做成一首歌" to generate

The app will display the audio and structured lyrics.

In [ ]:
import os
os.chdir("/content/music")

import app

print("Launching Gradio app...")
print("This will start an interactive interface accessible via a public share link.\n")

# Launch with share=True for public access
app.demo.launch(share=True)

## Notes

- **GPU Memory:** Monitor VRAM on first run. The model weights (~2-3GB) plus inference can be memory-intensive.
- **Anthropic API:** Ensure your API key is valid and has sufficient credits.
- **Share Link Timeout:** Gradio share links expire after 72 hours. Re-run this cell to generate a new one.
- **Offline Mode:** If you want to run locally instead, clone the repo and run `python app.py`